[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vladimiracunadev-create/python-data-science-bootcamp/blob/master/classes/10-modelos-supervisados/notebook.ipynb)

# Clase 10 — Modelos Supervisados: Clasificacion

---

## Objetivos de la clase

Al terminar esta clase seras capaz de:

- **Entender** la diferencia entre clasificacion y regresion
- **Entrenar** un `DecisionTreeClassifier` paso a paso
- **Interpretar** la matriz de confusion y sus cuatro cuadrantes
- **Comparar** el arbol de decision con `LogisticRegression`


## Concepto clave: Regresion vs Clasificacion

```
REGRESION vs CLASIFICACION
==========================

Regresion: predice un NUMERO CONTINUO
  Input: [antiguedad=24, productos=3]  ->  Output: 45200 (ventas predichas)
                                                    ^^^^^
                                               numero real, sin limite

Clasificacion: predice una CATEGORIA
  Input: [antiguedad=24, productos=3]  ->  Output: CHURN=1 (cliente se va)
                                                   CHURN=0 (cliente se queda)
                                                   ^^^^^^^
                                              una de N categorias posibles

Ejemplos de clasificacion:
  - Email:      SPAM / NO SPAM                    (2 clases = binario)
  - Diagnostico: sano / enfermedad_A / enfermedad_B (3 clases = multiclase)
  - Credito:    aprobado / rechazado              (2 clases = binario)
```


In [ ]:
# ============================================================
# BLOQUE: Importacion de librerias
# ============================================================

# --- Manipulacion y calculo numerico ---
import pandas as pd          # pandas: tablas de datos (DataFrames)
import numpy as np           # numpy: operaciones matematicas vectorizadas

# --- Visualizacion ---
import matplotlib.pyplot as plt  # pyplot: API de graficos estilo MATLAB

# --- Modelos de clasificacion ---
from sklearn.tree import DecisionTreeClassifier      # arbol de decision binario
from sklearn.linear_model import LogisticRegression  # clasificador lineal con sigmoide

# --- Particion y validacion ---
from sklearn.model_selection import train_test_split  # dividir datos en train/test
from sklearn.model_selection import cross_val_score   # validacion cruzada K-Fold

# --- Metricas de evaluacion ---
from sklearn.metrics import confusion_matrix          # matriz 2x2 de errores
from sklearn.metrics import ConfusionMatrixDisplay    # grafico profesional de la matriz
from sklearn.metrics import classification_report     # precision, recall, f1 por clase
from sklearn.metrics import f1_score                  # metrica F1 individual

# --- Preprocesamiento ---
from sklearn.preprocessing import StandardScaler  # escalar a media=0, desviacion=1

# Verificar que todo cargo correctamente
print("Librerias cargadas correctamente")  # debe imprimirse sin errores


## Seccion 1: Cargar y explorar `retencion_clientes.csv`

Antes de entrenar cualquier modelo debemos conocer los datos: cuantos son, que columnas tienen, y como esta distribuido el target.


In [ ]:
# ============================================================
# BLOQUE: Carga y exploracion del dataset
# ============================================================

# Paso 1: leer el archivo CSV y convertirlo en DataFrame
# pd.read_csv detecta el separador y los tipos de columna automaticamente
df = pd.read_csv("../../datasets/retencion_clientes.csv")  # cargar CSV desde ruta relativa

# Paso 2: verificar cuantos registros y columnas cargamos
# .shape devuelve la tupla (n_filas, n_columnas)
print(f"Dimensiones del dataset: {df.shape}")  # Ejemplo esperado: (500, 6)
print()  # separador visual

# Paso 3: ver las primeras 3 filas para entender la estructura
# head(n) muestra los n primeros registros
print("Primeras 3 filas del dataset:")
print(df.head(3))  # mostrar muestra de datos
print()

# Paso 4: contar cuantos clientes de cada clase hay en el target
# value_counts() cuenta las ocurrencias de cada valor unico
print("Distribucion de la variable objetivo (churned):")
conteo = df["churned"].value_counts()  # conteo absoluto por clase
print(conteo)
print()

# Paso 5: calcular el porcentaje de cada clase
# normalize=True convierte conteos a proporciones (0.0 a 1.0)
# multiplicar x100 para obtener porcentajes legibles
pct = df["churned"].value_counts(normalize=True) * 100  # porcentajes por clase
print("Porcentaje por clase:")
print(pct.round(1))  # redondear a 1 decimal para mayor legibilidad


## Estructura del dataset

```
DATASET: retencion_clientes.csv
================================

| antiguedad_meses | n_productos | reclamos_ultimo_ano | segmento | churned |
|------------------|-------------|---------------------|----------|---------|
|        36        |      2      |          1          |  premium |    0    |
|        12        |      1      |          4          |  basico  |    1    |
|        60        |      3      |          0          |  premium |    0    |
        ^               ^                 ^                ^           ^
   FEATURES (X): variables predictoras                      TARGET (y)
                                                        0 = se quedo
                                                        1 = se fue (churn)

El modelo aprende la relacion:
  f(antiguedad, n_productos, reclamos) -> churned
```


In [ ]:
# ============================================================
# BLOQUE: Seleccion de features y definicion de X e y
# ============================================================

# Paso 1: identificar columnas numericas automaticamente
# select_dtypes(include=["number"]) filtra solo columnas int64 y float64
cols_num = df.select_dtypes(include=["number"]).columns.tolist()  # lista de nombres numericos
print(f"Columnas numericas encontradas: {cols_num}")

# Paso 2: construir la lista de features excluyendo el target
# Usamos comprension de lista para filtrar 'churned' de la lista numerica
features = [c for c in cols_num if c != "churned"]  # solo predictores, sin target
print(f"Features seleccionadas: {features}")

# Paso 3: crear la matriz de features X
# X es un DataFrame con forma (n_muestras, n_features)
X = df[features]  # seleccionar solo las columnas predictoras

# Paso 4: crear el vector target y
# y es una Serie de pandas con valores 0 (quedo) o 1 (se fue)
y = df["churned"]  # columna que queremos predecir

# Paso 5: verificar dimensiones para confirmar que todo esta bien
print(f"\nShape de X (features):  {X.shape}")
print(f"Shape de y (target):    {y.shape}")
print(f"Columnas en X: {list(X.columns)}")


In [ ]:
# ============================================================
# BLOQUE: Division en conjuntos de entrenamiento y prueba
# ============================================================

# Paso 1: dividir datos con train_test_split
# test_size=0.2  -> reservar el 20% de los datos para evaluar el modelo
# random_state=42 -> fijar la semilla para obtener el mismo split siempre
# stratify=y     -> garantizar que la proporcion de 0s y 1s es igual en train y test
X_train, X_test, y_train, y_test = train_test_split(
    X,              # features completas
    y,              # target completo
    test_size=0.2,  # 20% para evaluacion (80% para entrenamiento)
    random_state=42, # reproducibilidad: mismo resultado en cada ejecucion
    stratify=y      # preservar balance de clases en ambas particiones
)

# Paso 2: confirmar el tamano de cada conjunto
print(f"Conjunto de entrenamiento: {X_train.shape[0]} filas ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Conjunto de prueba:        {X_test.shape[0]} filas ({X_test.shape[0]/len(X)*100:.0f}%)")

# Paso 3: verificar el balance de clases en cada particion
# Si stratify funciono, los porcentajes deben ser iguales
print("\nBalance de clases en TRAIN:")
print((y_train.value_counts(normalize=True) * 100).round(1))  # porcentajes en train
print("\nBalance de clases en TEST:")
print((y_test.value_counts(normalize=True) * 100).round(1))   # porcentajes en test


## Como funciona un Arbol de Decision

```
ARBOL DE DECISION — Como funciona
===================================

         Pregunta raiz: antiguedad_meses > 24?
                       /                    \
                     SI                      NO
              reclamos > 2?          n_productos > 1?
              /         \              /             \
          CHURN=1    CHURN=0       CHURN=1         CHURN=0
          (se va)  (se queda)    (se va)          (se queda)

Cada nodo hace UNA pregunta binaria sobre UNA feature.
El algoritmo busca el MEJOR corte en cada nivel
(el que mas reduce la impureza de Gini o la entropia).

PARAMETROS CLAVE:
  max_depth        = cuantos niveles de preguntas (mas niveles = mas complejo)
  min_samples_leaf = minimo de filas en cada hoja (evita over-fitting)

RIESGO: si el arbol es muy profundo, memoriza los datos de entrenamiento
        y falla en datos nuevos  ->  OVERFITTING
```


In [ ]:
# ============================================================
# BLOQUE: Entrenamiento del Arbol de Decision
# ============================================================

# Paso 1: instanciar el clasificador con sus hiperparametros
# max_depth=4      -> maximo 4 niveles de preguntas (balance entre precision y generalizacion)
# min_samples_leaf=5 -> cada hoja final debe tener al menos 5 muestras (evita sobreajuste)
# random_state=42  -> reproducibilidad cuando hay empates en la impureza de Gini
arbol = DecisionTreeClassifier(
    max_depth=4,           # limitar profundidad para generalizar mejor a datos nuevos
    min_samples_leaf=5,    # no crear nodos hoja con menos de 5 ejemplos
    random_state=42        # misma ejecucion en cada corrida del notebook
)

# Paso 2: entrenar el modelo con los datos de entrenamiento
# .fit(X, y) es el metodo universal de sklearn para entrenar
# Internamente calcula la mejor pregunta binaria para cada nodo
arbol.fit(X_train, y_train)  # el modelo aprende las reglas de decision

# Paso 3: generar predicciones sobre el conjunto de prueba
# .predict() recorre el arbol aprendido y devuelve la clase en cada hoja
y_pred_arbol = arbol.predict(X_test)  # array de 0s y 1s para cada fila de X_test

# Paso 4: medir la exactitud del modelo
# .score(X, y) = porcentaje de predicciones correctas sobre el total
exactitud = arbol.score(X_test, y_test)  # comparar predicciones con etiquetas reales
print(f"Exactitud del Arbol de Decision: {exactitud:.4f} ({exactitud*100:.1f}%)")
# NOTA: la exactitud sola puede enganar si las clases estan desbalanceadas


## Seccion 2: Matriz de Confusion

La exactitud (accuracy) sola puede ser enganosa. La **matriz de confusion** nos dice exactamente donde falla el modelo.

```
MATRIZ DE CONFUSION
====================

                  PREDICHO: NO churn   PREDICHO: SI churn
REAL: NO churn  [  TN (correcto)     |  FP (falsa alarma)  ]
REAL: SI churn  [  FN (error grave)  |  TP (correcto)      ]

TN = True Negative   | Predijo NO churn, y era NO churn   OK
TP = True Positive   | Predijo SI churn, y era SI churn   OK
FP = False Positive  | Predijo SI churn, pero era NO churn  ERROR (falsa alarma)
FN = False Negative  | Predijo NO churn, pero era SI churn  ERROR GRAVE para negocio

Para retencion de clientes:
  FN es el peor error -> cliente que SE VA pero el modelo no lo detecta
  -> perdemos la oportunidad de retenerlo con una oferta especial
```


In [ ]:
# ============================================================
# BLOQUE: Visualizacion de la Matriz de Confusion
# ============================================================

# Paso 1: calcular la matriz numerica
# confusion_matrix(reales, predicciones) devuelve un array 2x2:
# [[TN, FP],
#  [FN, TP]]
cm = confusion_matrix(y_test, y_pred_arbol)  # matriz de errores y aciertos
print("Matriz de confusion (valores crudos):")
print(cm)  # imprimir la matriz numerica para referencia
print()

# Paso 2: extraer los cuatro valores individualmente
# .ravel() aplana la matriz 2x2 a un array de 4 elementos en orden: TN, FP, FN, TP
tn, fp, fn, tp = cm.ravel()  # desempaquetar los cuatro cuadrantes
print(f"  TN (verdadero negativo): {tn} — predijo no churn y era no churn")
print(f"  FP (falso positivo):     {fp} — predijo churn pero era no churn")
print(f"  FN (falso negativo):     {fn} — predijo no churn pero ERA churn  <-- critico")
print(f"  TP (verdadero positivo): {tp} — predijo churn y era churn")
print()

# Paso 3: crear el lienzo del grafico
fig, ax = plt.subplots(figsize=(6, 5))  # figura de 6x5 pulgadas

# Paso 4: construir el objeto de visualizacion
# ConfusionMatrixDisplay transforma la matriz numerica en un heatmap legible
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,                          # los datos numericos
    display_labels=["NO churn (0)", "SI churn (1)"]  # etiquetas de eje
)

# Paso 5: dibujar el heatmap
# cmap='Blues' usa una escala de azules (mas oscuro = mas frecuente)
disp.plot(ax=ax, cmap="Blues", colorbar=False)  # sin barra de color lateral

# Paso 6: agregar titulo informativo
ax.set_title("Matriz de Confusion — Arbol de Decision", fontsize=13, fontweight="bold")

plt.tight_layout()  # ajustar margenes automaticamente
plt.show()          # renderizar el grafico


In [ ]:
# ============================================================
# BLOQUE: Reporte de Clasificacion completo
# ============================================================

# El classification_report calcula 4 metricas por cada clase:
#
# PRECISION  = TP / (TP + FP)
#   De todos los que predije como churners, cuantos realmente lo eran?
#   Alta precision -> pocas falsas alarmas (no molestamos a clientes que no se iban)
#
# RECALL (sensibilidad) = TP / (TP + FN)
#   De todos los churners reales, cuantos detecte correctamente?
#   Alto recall -> pocos churners se escaping sin ser detectados (prioridad de negocio)
#
# F1-SCORE = 2 * (precision * recall) / (precision + recall)
#   Media armonica entre precision y recall
#   Es mas conservadora que el promedio: si uno es muy bajo, el F1 baja mucho
#   Util especialmente cuando las clases estan desbalanceadas
#
# SUPPORT = cuantos ejemplos reales de esa clase hay en el test set

print("Reporte de Clasificacion — Arbol de Decision:")
print("=" * 55)
print(
    classification_report(
        y_test,          # etiquetas reales del conjunto de prueba
        y_pred_arbol,    # predicciones del modelo
        target_names=["NO churn", "SI churn"]  # nombres legibles por clase
    )
)


## Seccion 3: Regresion Logistica

La Regresion Logistica usa la funcion **sigmoide** para convertir un puntaje lineal en una probabilidad entre 0 y 1:

```
FUNCION SIGMOIDE — Convierte puntajes lineales en probabilidades
================================================================

                        1
  P(churn=1) =  -------------------
                  1 + e^(-(w*X + b))

Forma caracteristica de S:

  1.0 |                              ______
      |                         ____/
  0.5 |------------------------/------------ <- UMBRAL (P > 0.5 -> clase 1)
      |                   ____/
  0.0 |__________________/
      +------------------------------------------------>
                      w*X + b

IMPORTANTE: la Regresion Logistica es sensible a la escala de features.
  Si edad va de 0-100 y saldo va de 0-100000, el saldo dominara el aprendizaje.
  Solucion: StandardScaler convierte cada feature a media=0, desviacion=1
```


In [ ]:
# ============================================================
# BLOQUE: Regresion Logistica con StandardScaler
# ============================================================

# Paso 1: instanciar el escalador
# StandardScaler aprende la media y desviacion de cada feature en el train set
# y luego aplica la formula: X_scaled = (X - media) / desviacion
scaler = StandardScaler()  # crear objeto escalador (aun no ha aprendido nada)

# Paso 2: ajustar el escalador SOLO con datos de entrenamiento
# REGLA DE ORO: el escalador nunca debe ver datos de test antes de predecir
# fit_transform: calcula media/std del train Y transforma el train en un paso
X_train_sc = scaler.fit_transform(X_train)  # aprender escala y transformar train

# Paso 3: transformar el test usando la escala aprendida del train
# Solo .transform() (sin .fit()) -> usamos la media/std aprendida del train
# Si hicieramos fit en test, habria DATA LEAKAGE (el modelo ve estadisticas del test)
X_test_sc = scaler.transform(X_test)  # aplicar la misma escala del train al test

# Paso 4: instanciar y entrenar la Regresion Logistica
# max_iter=1000 -> aumentar iteraciones del optimizador para asegurar convergencia
# random_state=42 -> reproducibilidad (afecta al solver cuando hay aleatoriedad)
rl = LogisticRegression(
    max_iter=1000,   # iteraciones del optimizador (default=100 puede no converger)
    random_state=42  # reproducibilidad
)
rl.fit(X_train_sc, y_train)  # entrenar con las features escaladas

# Paso 5: generar predicciones sobre el test escalado
y_pred_rl = rl.predict(X_test_sc)  # clases predichas: 0 o 1

# Paso 6: mostrar el reporte completo de la Regresion Logistica
print("Reporte de Clasificacion — Regresion Logistica:")
print("=" * 55)
print(
    classification_report(
        y_test,      # etiquetas reales
        y_pred_rl,   # predicciones de la RL
        target_names=["NO churn", "SI churn"]
    )
)


In [ ]:
# ============================================================
# BLOQUE: Comparacion de ambos modelos
# ============================================================

# Paso 1: calcular accuracy de cada modelo
# .score(X, y) calcula el porcentaje de predicciones correctas
acc_arbol = arbol.score(X_test, y_test)     # accuracy del arbol (sin escalar)
acc_rl    = rl.score(X_test_sc, y_test)     # accuracy de RL (con features escaladas)

# Paso 2: calcular F1 ponderado de cada modelo
# average='weighted' pondera el F1 de cada clase por su frecuencia real (support)
# Es mas justo que 'macro' cuando las clases estan desbalanceadas
f1_arbol = f1_score(y_test, y_pred_arbol, average="weighted")  # F1 del arbol
f1_rl    = f1_score(y_test, y_pred_rl,    average="weighted")  # F1 de la RL

# Paso 3: construir un diccionario con los resultados
# Estructura: {nombre_modelo: {metrica: valor}}
resultados = {
    "Arbol de Decision":   {"Accuracy": round(acc_arbol, 4), "F1 (weighted)": round(f1_arbol, 4)},
    "Regresion Logistica": {"Accuracy": round(acc_rl,    4), "F1 (weighted)": round(f1_rl,    4)},
}

# Paso 4: convertir el diccionario a DataFrame para mostrar como tabla
# orient='index' usa las claves del dict como filas de la tabla
df_comp = pd.DataFrame.from_dict(resultados, orient="index")  # tabla comparativa

# Paso 5: mostrar la tabla
print("Comparacion de modelos:")
print("=" * 45)
print(df_comp.to_string())  # .to_string() evita el formato truncado de Jupyter
print()

# Determinar el mejor modelo por F1
mejor = df_comp["F1 (weighted)"].idxmax()  # nombre de la fila con mayor F1
print(f"Modelo recomendado (mayor F1): {mejor}")
print("NOTA: Para retencion de clientes, priorizar el RECALL de la clase 1 (churners).")


## EJERCICIO

Agrega `reclamos_ultimo_ano` como feature adicional y re-entrena ambos modelos.

**Preguntas a responder:**
1. Despues de agregar la feature, ¿mejora el F1-score del Arbol de Decision?
2. ¿Cual modelo se beneficia mas de esta nueva feature?

```python
# TU CODIGO AQUI
# Paso 1: redefinir features_seleccionadas incluyendo 'reclamos_ultimo_ano'
# Paso 2: re-crear X con las nuevas features
# Paso 3: re-ejecutar train_test_split con la nueva X
# Paso 4: re-entrenar el DecisionTreeClassifier
# Paso 5: re-escalar y re-entrenar LogisticRegression
# Paso 6: comparar las metricas con la tabla de resultados anterior
```


## Resumen de la Clase 10

**Checklist — lo que aprendiste hoy:**

- [ ] Diferencia entre regresion (numero continuo) y clasificacion (categoria discreta)
- [ ] Como preparar features (X) y target (y) para un problema de clasificacion
- [ ] Como dividir datos en train/test con `train_test_split` y `stratify=y`
- [ ] Como entrenar `DecisionTreeClassifier` con `max_depth` y `min_samples_leaf`
- [ ] Como leer la **matriz de confusion**: TN, TP, FP, FN y su impacto de negocio
- [ ] Diferencia entre **precision** (pocas alarmas falsas) y **recall** (detectar todos los churners)
- [ ] Por que se necesita `StandardScaler` antes de `LogisticRegression`
- [ ] Como comparar modelos en una tabla usando accuracy y F1-score

**Proxima clase:** Evaluacion robusta con Cross-Validation y Pipelines de ML.
